In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems08.npz")["data"]

traffic = data[:, :, 0]

print(traffic.shape)

(17856, 170)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(17856, 170)


In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(17844, 12, 170)
(17844, 170)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(14275, 12, 170)
(3569, 12, 170)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn

class MultiScaleTemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv3 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

        self.conv5 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(5,1),
            padding=(2,0)
        )

        self.conv7 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(7,1),
            padding=(3,0)
        )

    def forward(self,x):

        out3 = self.conv3(x)

        out5 = self.conv5(x)

        out7 = self.conv7(x)

        return torch.relu(
            out3 + out5 + out7
        )

In [9]:
class GraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [10]:
class MultiScaleSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = MultiScaleTemporalConv(
            1,
            32
        )

        self.graph = GraphConv(
            num_nodes=170,
            channels=32
        )

        self.temp2 = MultiScaleTemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.graph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [11]:
model = MultiScaleSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 170])
torch.Size([64, 170])


In [12]:
model = MultiScaleSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.014471
Epoch 2/30 Loss: 0.002578
Epoch 3/30 Loss: 0.002448
Epoch 4/30 Loss: 0.002414
Epoch 5/30 Loss: 0.002380
Epoch 6/30 Loss: 0.002360
Epoch 7/30 Loss: 0.002361
Epoch 8/30 Loss: 0.002353
Epoch 9/30 Loss: 0.002356
Epoch 10/30 Loss: 0.002347
Epoch 11/30 Loss: 0.002348
Epoch 12/30 Loss: 0.002352
Epoch 13/30 Loss: 0.002335
Epoch 14/30 Loss: 0.002345
Epoch 15/30 Loss: 0.002358
Epoch 16/30 Loss: 0.002340
Epoch 17/30 Loss: 0.002330
Epoch 18/30 Loss: 0.002330
Epoch 19/30 Loss: 0.002328
Epoch 20/30 Loss: 0.002348
Epoch 21/30 Loss: 0.002334
Epoch 22/30 Loss: 0.002331
Epoch 23/30 Loss: 0.002320
Epoch 24/30 Loss: 0.002318
Epoch 25/30 Loss: 0.002302
Epoch 26/30 Loss: 0.002324
Epoch 27/30 Loss: 0.002326
Epoch 28/30 Loss: 0.002303
Epoch 29/30 Loss: 0.002290
Epoch 30/30 Loss: 0.002284


In [13]:
train_time = time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 2059.453590154648


In [14]:
torch.save(
    model.state_dict(),
    "MultiScaleSTGCN-PEMS08.pth"
)

In [15]:
model.eval()

infer_start = time.time()

with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 8.17491865158081
MAE: 0.032294426
RMSE: 0.04631905


In [16]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 10765.628051757812
R2: 0.9653583860992333


In [17]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 19841
